# Task 2.2 – Real-time Violation Visualisation

This notebook connects to MongoDB and continuously polls for new violation records
written by the streaming application. The plots update in real-time each time a
new micro-batch lands in the database.

Run this notebook **at the same time** as `data_design_streaming.ipynb` to see
the graphs update live as violations are detected.

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `POLL_INTERVAL` | 5 s | How often MongoDB is queried |
| `WINDOW` | 20 pts | Rolling window – oldest points are dropped |
| Moving avg window | 5 pts | Same as Week-10 Scenario 4 class example |
| Spike threshold | 1.5 × mean | Flags sudden count surges in `shade_spikes()` |
| Percentile level | 90th | Dynamically computed per frame on current window |


## Setup

In [ ]:
from time import sleep
from datetime import datetime
import statistics

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from pymongo import MongoClient

# needed for real-time interactive display in Jupyter Notebook
%matplotlib notebook

HOST_IP       = 'host.docker.internal'
POLL_INTERVAL = 5    # seconds between MongoDB queries
WINDOW        = 20   # how many poll points to keep in the rolling window


## How it works

The main loop mirrors the Week-10 consumer pattern:

```
while True:
    docs = query_new_violations(collection, last_poll_time)
    # aggregate this batch → inst count, avg count, mean speed
    # append to rolling lists
    # redraw both subplots
    fig.canvas.draw()   # ← same as Week-10 consumer update
    x.pop(0)            # ← sliding window, same as Week-10
    sleep(POLL_INTERVAL)
```

MongoDB is queried with an aggregation pipeline that unwinds the nested
`violations` array and filters records where `timestamp_start > last_poll_time`.


## Task 2.2.1 & 2.2.2 – Visualisation

In [ ]:
# ─── Annotation helpers (based on Week-10 Scenario 2 / 4 class code) ─────────

def annotate_max(x, y, ax=None):
    ymax = max(y)
    xpos = y.index(ymax)
    xmax = x[xpos]
    text = 'Max: Time={}, Value={}'.format(xmax.strftime('%H:%M:%S'), round(ymax, 1))
    if not ax:
        ax = plt.gca()
    offset = max(y) * 0.12 + 0.1
    ax.annotate(text, xy=(xmax, ymax), xytext=(xmax, ymax + offset),
                arrowprops=dict(facecolor='red', shrink=0.05))

def annotate_min(x, y, ax=None):
    ymin = min(y)
    xpos = y.index(ymin)
    xmin = x[xpos]
    text = 'Min: Time={}, Value={}'.format(xmin.strftime('%H:%M:%S'), round(ymin, 1))
    if not ax:
        ax = plt.gca()
    offset = max(y) * 0.12 + 0.1
    ax.annotate(text, xy=(xmin, ymin), xytext=(xmin, ymin + offset),
                arrowprops=dict(facecolor='orange', shrink=0.05))

# dynamically computes and labels the Nth percentile (HD requirement)
def annotate_percentile(x, y, ax, pct=90):
    p_val = float(np.percentile(y, pct))
    ax.axhline(y=p_val, color='purple', linestyle=':', linewidth=1.5,
               label='P{} = {:.1f} km/h'.format(pct, p_val))
    ax.text(x[0], p_val + 0.5, ' P{} = {:.1f}'.format(pct, p_val),
            color='purple', fontsize=8)

# shades time windows where the value spikes above factor * mean (Distinction)
def shade_spikes(x, y, ax, factor=1.5):
    mean_val = statistics.mean(y)
    if mean_val == 0:
        return
    threshold = mean_val * factor
    labeled = False
    for i, yi in enumerate(y):
        if yi > threshold:
            xs = x[max(0, i - 1)]
            xe = x[min(len(x) - 1, i + 1)]
            lbl = 'Spike Region' if not labeled else ''
            ax.axvspan(xs, xe, alpha=0.15, color='red', label=lbl)
            labeled = True


# ─── MongoDB helpers ──────────────────────────────────────────────────────────

def connect_mongo():
    try:
        client = MongoClient(host=HOST_IP, port=27017)
        return client
    except Exception as e:
        print('MongoDB connection failed:', e)
        return None

def query_new_violations(collection, since):
    # unwind the nested violations array and filter by timestamp_start
    pipeline = [
        {'$unwind': '$violations'},
        {'$match': {'violations.timestamp_start': {'$gt': since}}},
        {'$project': {
            '_id': 0,
            'violation_type': '$violations.violation_type',
            'speed_reading':  '$violations.speed_reading',
        }}
    ]
    return list(collection.aggregate(pipeline))


# ─── Plot initialisation (same pattern as Week-10 Consumer 3) ────────────────

def init_plots():
    fig = plt.figure(figsize=(11, 7))
    fig.subplots_adjust(hspace=0.7)

    ax1 = fig.add_subplot(211)
    ax1.set_xlabel('Arrival Time')
    ax1.set_ylabel('Violation Count')
    ax1.title.set_text('Violation Count vs Arrival Time')

    ax2 = fig.add_subplot(212)
    ax2.set_xlabel('Arrival Time')
    ax2.set_ylabel('Speed (km/h)')
    ax2.title.set_text('Speed Pattern vs Arrival Time')

    fig.suptitle('AWAS Real-time Violation Dashboard')
    fig.show()
    fig.canvas.draw()
    return fig, ax1, ax2


# ─── Main real-time loop ──────────────────────────────────────────────────────

def visualize_violations():
    client = connect_mongo()
    if client is None:
        return
    collection = client['traffic_monitoring']['violations']

    fig, ax1, ax2 = init_plots()

    # rolling window containers
    x, y_inst, y_avg, y_speed, y_moving = [], [], [], [], []

    # start from the beginning of the camera event dataset
    last_poll = datetime(2024, 1, 1)

    try:
        while True:
            now  = datetime.now()
            docs = query_new_violations(collection, last_poll)
            last_poll = now

            # aggregate this poll window
            inst  = sum(1 for d in docs if d['violation_type'] == 'INSTANTANEOUS')
            avg   = sum(1 for d in docs if d['violation_type'] == 'AVERAGE')
            spds  = [d['speed_reading'] for d in docs if d['speed_reading'] > 0]
            mean_spd = statistics.mean(spds) if spds else 0

            x.append(now)
            y_inst.append(inst)
            y_avg.append(avg)
            y_speed.append(mean_spd)

            # 5-window moving average – same as Week-10 Scenario 4
            if len(y_speed) > 5:
                y_moving.append(statistics.mean(y_speed[-5:]))
            else:
                y_moving.append(statistics.mean(y_speed) if y_speed else 0)

            # start drawing once we have enough data points
            if len(x) > 10:

                # ── subplot 1: violation counts (Task 2.2.1) ──
                ax1.clear()
                ax1.plot(x, y_inst, marker='o', linestyle='-',
                         color='crimson',   linewidth=1.8, markersize=5,
                         label='Instantaneous')
                ax1.plot(x, y_avg,  marker='s', linestyle='--',
                         color='steelblue', linewidth=1.8, markersize=5,
                         label='Average Speed')
                ax1.set_xlabel('Arrival Time')
                ax1.set_ylabel('Violation Count')
                ax1.set_title('Violation Count vs Arrival Time')
                ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
                ax1.tick_params(axis='x', rotation=30)
                ax1.legend(loc='upper right', fontsize=8)

                # Task 2.2.2 annotations on ax1
                annotate_max(x, y_inst, ax1)        # max callout (Pass)
                annotate_min(x, y_inst, ax1)        # min callout (Credit)
                shade_spikes(x, y_inst, ax1, 1.5)  # spike shading (Distinction)

                # ── subplot 2: speed pattern (Task 2.2.1) ────
                ax2.clear()
                ax2.plot(x, y_speed,  marker='.', linestyle='-',
                         color='salmon', linewidth=0.8, alpha=0.6,
                         label='Mean Speed (per poll)')
                ax2.plot(x, y_moving, marker='^', linestyle='-',
                         color='navy',   linewidth=2.0, markersize=5,
                         label='Moving Avg (5-win)')
                # camera speed limit reference lines
                ax2.axhline(y=110, color='red',        linestyle='-.',
                            linewidth=1.2, alpha=0.7, label='Limit Cam 1&2 (110 km/h)')
                ax2.axhline(y=90,  color='darkorange', linestyle='-.',
                            linewidth=1.2, alpha=0.7, label='Limit Cam 3 (90 km/h)')
                ax2.set_xlabel('Arrival Time')
                ax2.set_ylabel('Speed (km/h)')
                ax2.set_title('Speed Pattern vs Arrival Time')
                ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
                ax2.tick_params(axis='x', rotation=30)

                # Task 2.2.2 annotations on ax2
                annotate_max(x, y_moving, ax2)           # max callout (Pass)
                annotate_min(x, y_moving, ax2)           # min callout (Credit)
                annotate_percentile(x, y_moving, ax2, 90)  # 90th pct (HD)
                ax2.legend(loc='upper right', fontsize=8)

                fig.canvas.draw()

                # slide the window – drop the oldest point
                x.pop(0)
                y_inst.pop(0)
                y_avg.pop(0)
                y_speed.pop(0)
                y_moving.pop(0)

            sleep(POLL_INTERVAL)

    except KeyboardInterrupt:
        print('Stopped.')
    finally:
        client.close()
        plt.close('all')


if __name__ == '__main__':
    visualize_violations()


## Interesting Points – Operational Significance

The annotations added to each frame explain *why* a particular moment matters:

**`annotate_max` / `annotate_min`** — pinpoints the highest and lowest violation
activity in the current window. The peak tells enforcement teams which minute
to focus on; the trough shows when the road is quietest.

**`shade_spikes`** — shades any time bucket where the count exceeds 1.5× the
window mean. A sudden surge may indicate an upstream incident causing bunched
traffic, or a particularly aggressive batch of drivers entering the segment.

**`annotate_percentile(pct=90)`** — draws a dynamically computed 90th-percentile
line on the speed plot. Vehicles consistently above this line are in the top 10%
of speeders in the current window, making them priority candidates for heavier
penalties under a tiered enforcement policy.

**Speed limit reference lines** — `axhline` at 110 km/h (cameras 1 & 2) and
90 km/h (camera 3) gives an immediate visual benchmark: any `moving_avg` above
the dashed line represents a sustained exceedance, not just a momentary spike.
